# Rice Leaf Disease Classification

Train a convolutional neural network to classify rice leaf diseases using the `Project-AgML/uci_rice_leaf_diseases_classification` dataset. Run the cells from top to bottom. The required packages are NumPy, Hugging Face `datasets`, and TensorFlow.

## 1. Imports and configuration

In [ ]:
import numpy as np
from datasets import load_dataset
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 128

## 2. Load the dataset

In [ ]:
ds = load_dataset("Project-AgML/uci_rice_leaf_diseases_classification")
data = ds[list(ds.keys())[0]]
class_names = data.features["label"].names

print("Classes:", class_names)
print("Number of images:", len(data))

## 3. Prepare the images and labels

Convert each image to RGB, resize it to `128 × 128`, and scale pixel values to the `[0, 1]` range.

In [ ]:
X = np.array(
    [
        np.array(img.convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        for img in data["image"]
    ],
    dtype="float32",
) / 255.0
y = np.array(data["label"])

print("Image array shape:", X.shape)
print("Label array shape:", y.shape)

## 4. Shuffle and split into training and test sets

Use 80% of the images for training and 20% for testing.

In [ ]:
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

print("Training images:", len(X_train))
print("Test images:", len(X_test))

## 5. Build and compile the CNN

Random flipping and rotation are applied as lightweight data augmentation during training.

In [ ]:
model = keras.Sequential(
    [
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(class_names), activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 6. Train the model

The test split is also used for validation during training, matching the original script.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_test, y_test),
)

## 7. Evaluate and predict

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.2f}")

In [ ]:
pred = model.predict(X_test[:1])
print("Predicted disease:", class_names[np.argmax(pred)])
print("Actual disease:   ", class_names[y_test[0]])

## 8. Save the trained model

This writes `rice_disease_model.keras` to the notebook's current working directory. Run the notebook with the project folder as the working directory to save it beside this notebook.

In [ ]:
model.save("rice_disease_model.keras")